In [1]:
from pathlib import Path
import sys

repo = Path("/Users/zombak/git/compresso")
sys.path.insert(0, str(repo / "examples"))

import compresso.clustering as cc
from compresso.io import load_srp_tensor
from recsys_lib.checkpoint import read_checkpoint, load_recsys_split

In [2]:


checkpoint = repo / "artifacts/goodbooks/exp_descriptions.zip"

with read_checkpoint(checkpoint) as root:
    split = load_recsys_split(root)
    srp = load_srp_tensor(root / "sbert_sae" / "sparse_embeddings.srp.pt")

item_ids = split["item_ids"]
tags = split["entity_tag_matrix"]
tag_names = split["tag_names"]
meta = split["entity_metadata"]

print(srp.shape)
print(tags.shape if tags is not None else None)
print(tag_names[:20] if tag_names is not None else None)
meta.head()

(10000, 4096)
(10000, 2816)
['00-in-class' '02-fantasy' '04-realistic-fiction' '1' '100-books'
 '100-books-to-read-before-you-die' '100-books-to-read-in-a-lifetime'
 '1001' '1001-books' '1001-books-to-read'
 '1001-books-to-read-before-you-die' '1001-books-you-must-read-before-you'
 '1001-import' '1001-list' '1001-to-read' '14th-century' '16th-century'
 '17th-century' '1800s' '18th-century']


,item_id,title,authors,average_rating,description
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.34,"""Immerse yourself in the thrilling world of Th..."
1,10,Pride and Prejudice,Jane Austen,4.24,"""Experience the timeless romance and wit of Ja..."
2,100,The Poisonwood Bible,Barbara Kingsolver,4.02,"""Immerse yourself in the epic tale of The Pois..."
3,1000,"Shadow and Bone (Shadow and Bone, #1)",Leigh Bardugo,4.05,"""Immerse yourself in the epic fantasy world of..."
4,10000,The First World War,John Keegan,4.00,"""Experience the definitive account of the Grea..."


In [4]:
meta[meta.description.notna()]

,item_id,title,authors,average_rating,description
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.34,"""Immerse yourself in the thrilling world of Th..."
1,10,Pride and Prejudice,Jane Austen,4.24,"""Experience the timeless romance and wit of Ja..."
2,100,The Poisonwood Bible,Barbara Kingsolver,4.02,"""Immerse yourself in the epic tale of The Pois..."
3,1000,"Shadow and Bone (Shadow and Bone, #1)",Leigh Bardugo,4.05,"""Immerse yourself in the epic fantasy world of..."
4,10000,The First World War,John Keegan,4.00,"""Experience the definitive account of the Grea..."
...,...,...,...,...,...
9995,9995,"Billy Budd, Sailor",Herman Melville,3.09,"""Experience the classic tale of loyalty, honor..."
9996,9996,"Bayou Moon (The Edge, #2)",Ilona Andrews,4.09,"""Immerse yourself in the mystical world of The..."
9997,9997,"Means of Ascent (The Years of Lyndon Johnson, #2)",Robert A. Caro,4.25,"""Means of Ascent: The Years of Lyndon Johnson,..."
9998,9998,The Mauritius Command,Patrick O'Brian,4.35,"""Embark on a thrilling adventure with ""The Mau..."


In [99]:
checkpoint = repo / "artifacts/ml20m/exp_tags.zip"

with read_checkpoint(checkpoint) as root:
    split = load_recsys_split(root)
    srp = load_srp_tensor(root / "sae" / "sparse_embeddings.srp.pt")

item_ids = split["item_ids"]
tags = split["entity_tag_matrix"]
tag_names = split["tag_names"]
meta = split["entity_metadata"]

print(srp.shape)
print(tags.shape if tags is not None else None)
print(tag_names[:20] if tag_names is not None else None)
meta.head()

(20549, 4096)
(20549, 823)
['007' '01/11' '01/12' '02/11' '03/11' '04/11' '05/11' '06/11' '10/10'
 '100 essential female performances' '11/10' '11/11' '12/10' '12/11'
 '1930s' '1950s' '1960s' '1970s' '1980s' '19th century']


,item_id,title,genres,description
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Get ready for a rootin' tootin' good time with...
1,10,GoldenEye (1995),Action|Adventure|Thriller,"""Experience the high-stakes action and espiona..."
2,100,City Hall (1996),Drama|Thriller,"""Experience the gripping drama of City Hall, a..."
3,1000,Curdled (1996),Crime,"""Uncover the dark secrets of the past with Cur..."
4,100003,Up in Smoke (1957),Comedy,"""Get ready for a wild ride with the 1957 comed..."


In [3]:
meta.iloc[2815,-1]

'"Experience the heartwarming and inspiring story of Marty and her journey to the American frontier in "Love Comes Softly" by Janette Oke. After her husband\'s untimely death, Marty is left alone and pregnant, forced to marry the kind but rugged Clark Davis to secure a future for herself and her unborn child. As she navigates her new role as a stepmother to Clark\'s young daughter, Missie, Marty must confront her own doubts and fears, learning to find wholeness and love through patience, faith, and the power of human connection. A classic tale of hope, resilience, and the transformative power of love, "Love Comes Softly" is a timeless and captivating read that will leave you feeling uplifted and inspired."'

In [71]:
meta[meta.title.str.contains("ermin")]

,item_id,title,genres
1551,110026,Prison Terminal: The Last Days of Private Jack...,Crime|Documentary
2815,1240,"Terminator, The (1984)",Action|Sci-Fi|Thriller
3711,1738,Vermin (1998),Comedy
5058,26333,"Terminal Man, The (1974)",Sci-Fi|Thriller
6857,32851,Hotel Terminus: The Life and Times of Klaus Ba...,Documentary
9952,4934,"Exterminator, The (1980)",Action|Crime|Thriller|War
11084,548,Terminal Velocity (1994),Action|Mystery|Thriller
11424,563,Germinal (1993),Drama|Romance
11961,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
12498,61018,Peppermint Candy (Bakha satang) (1999),Drama


In [3]:
meta = split["entity_metadata"]

# row index -> title
row_to_title = meta["title"].astype(str).to_numpy()

# title lowercase -> row indices
title_to_rows = {}
for i, title in enumerate(row_to_title):
    title_to_rows.setdefault(title.lower(), []).append(i)
def find_movies(query, limit=20):
    q = query.lower()
    hits = []
    for i, title in enumerate(row_to_title):
        if q in title.lower():
            hits.append((i, str(item_ids[i]), title, meta.iloc[i].get("genres", "")))
    return hits[:limit]

In [4]:
find_movies("Potter")

[(890,
  '18',
  'Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)',
  ''),
 (1112, '2', "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", ''),
 (1116, '2001', 'Harry Potter: Film Wizardry', ''),
 (1224,
  '21',
  'Harry Potter and the Order of the Phoenix (Harry Potter, #5)',
  ''),
 (1227, '2101', 'The Harry Potter Collection 1-4 (Harry Potter, #1-4)', ''),
 (1446,
  '23',
  'Harry Potter and the Chamber of Secrets (Harry Potter, #2)',
  ''),
 (1557, '24', 'Harry Potter and the Goblet of Fire (Harry Potter, #4)', ''),
 (1668, '25', 'Harry Potter and the Deathly Hallows (Harry Potter, #7)', ''),
 (1890, '27', 'Harry Potter and the Half-Blood Prince (Harry Potter, #6)', ''),
 (1941, '2745', "From Potter's Field (Kay Scarpetta, #6)", ''),
 (1990,
  '279',
  'Harry Potter and the Cursed Child - Parts One and Two (Harry Potter, #8)',
  ''),
 (2285,
  '3054',
  'Harry Potter and the Chamber of Secrets: Sheet Music for Flute with C.D',
  ''),
 (2530, '3275', 'Harry Pott

In [268]:
import importlib
importlib.reload(cc)

ImportError: cannot import name 'filter_clusters_by_size' from 'compresso.clustering.merge' (/Users/zombak/git/compresso/src/compresso/clustering/merge.py)

In [57]:
clusters_iou = cc.cluster_srp(
    srp,
    mode="combo_signed",
    top_m=,
    combo_size,
    min_cluster_size=5,
    #activation_iou_threshold=0.5,
    #post_merge_min_cluster_size=5
    #entity_tag_matrix=tags,
    #tag_names=tag_names,
    #top_k_tags=20,
    #tag_similarity_threshold=0.8,
)

In [24]:
from functools import partial

clusters = cc.run_clustering_pipeline(
    srp,
    [
        partial(
            cc.build_activation_clusters,
            mode="combo_signed",
            top_m=8,
            combo_size=2,
            min_cluster_size=5,
        ),
        lambda c: cc.merge_clusters_by_entity_iou(c, threshold=0.25),
    ],
    verbose=True,
)

[cluster_pipeline] step 1/2: build_activation_clusters
[cluster_pipeline] step 1/2 done: clusters=4689
[cluster_pipeline] step 2/2: <lambda> clusters_in=4689
[cluster_pipeline] step 2/2 done: clusters=952


In [14]:
def show_movie_clusters(query):
    hits = find_movies(query)
    cidss =[]
    for row_idx, item_id, title, genres in hits:
        print("\nMOVIE", row_idx, item_id, title, "|", genres)

        cluster_ids = clusters.entity_to_cluster_ids.get(row_idx, [])
        if not cluster_ids:
            print("  no cluster")
            continue
        cids = []
        for cid in cluster_ids:
            c = clusters.cluster_by_id[cid]
            cids.append(cid)
            print(" ", cid, "n=", c.entity_count, "features=", len(c.centroid.indices))
            print("   tags:", ", ".join(f"{t.name}:{t.score:.3f}" for t in c.tags[:5]))
        cidss.append(cids)
    return cidss
    

In [59]:
MOVIE 384 1343 The Light Fantastic (Discworld, #2; Rincewind #2) | 
  no cluster

MOVIE 377 1337 Going Postal (Discworld, #33; Moist von Lipwig, #1) | 
  combo:930:neg&1580:pos n= 35 features= 2
   tags: 

SyntaxError: invalid syntax (1267599386.py, line 1)

In [25]:
ll = show_movie_clusters("Discworld")


MOVIE 101 1089 Equal Rites (Discworld, #3; Witches #1) | 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 

MOVIE 377 1337 Going Postal (Discworld, #33; Moist von Lipwig, #1) | 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 

MOVIE 383 1342 Night Watch (Discworld, #29; City Watch, #6) | 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 

MOVIE 384 1343 The Light Fantastic (Discworld, #2; Rincewind #2) | 
  merge:merge_clusters_by_entity_iou:232 n= 10 features= 5
   tags: 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 

MOVIE 411 1368 Small Gods (Discworld, #13) | 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 12
   tags: 

MOVIE 805 1722 The Wee Free Men (Discworld, #30; Tiffany Aching, #1) | 
  merge:merge_clusters_by_entity_iou:233 n= 49 features= 1

In [26]:
show_cluster("merge:merge_clusters_by_entity_iou:233")


 merge:merge_clusters_by_entity_iou:233
n= 49
features= [(370, 0.2980000078678131), (960, 0.3709999918937683), (996, 0.06199999898672104), (1016, -0.12399999797344208), (1411, -0.1889999955892563), (1580, 0.40700000524520874), (2124, -0.3089999854564667), (2461, -0.06199999898672104), (2907, -0.09399999678134918), (3552, -0.1889999955892563), (3982, -0.5299999713897705), (4061, -0.35899999737739563)]
tags:
movies:
  101 1089 Equal Rites (Discworld, #3; Witches #1) | 
  377 1337 Going Postal (Discworld, #33; Moist von Lipwig, #1) | 
  383 1342 Night Watch (Discworld, #29; City Watch, #6) | 
  384 1343 The Light Fantastic (Discworld, #2; Rincewind #2) | 
  411 1368 Small Gods (Discworld, #13) | 
  805 1722 The Wee Free Men (Discworld, #30; Tiffany Aching, #1) | 
  855 1768 Wyrd Sisters (Discworld, #6; Witches #2) | 
  1029 1924 Reaper Man (Discworld, #11; Death, #2) | 
  1162 2043 Men at Arms (Discworld, #15; City Watch #2) | 
  1209 2086 Hogfather (Discworld, #20; Death, #4) | 
  1235 

In [27]:
show_cluster("merge:merge_clusters_by_entity_iou:234")


 merge:merge_clusters_by_entity_iou:234
n= 5
features= [(370, 0.289000004529953), (717, 0.8659999966621399), (1580, 0.289000004529953), (3982, -0.289000004529953)]
tags:
movies:
  1209 2086 Hogfather (Discworld, #20; Death, #4) | 
  1844 2658 Moving Pictures (Discworld, #10; Industrial Revolution, #1) | 
  2162 2944 The Truth (Discworld, #25; Industrial Revolution, #2) | 
  3177 3858 Interesting Times (Discworld, #17; Rincewind #5) | 
  5718 6144 Interesting Times: The Play | 


In [16]:
[s for s in [x for y in ll for x in y] if 'merge' in s]

['merge:merge_clusters_by_entity_iou:97']

In [37]:
len(meta[meta.authors.str.contains("Pratchet")])

50

In [17]:
show_cluster("merge:merge_clusters_by_entity_iou:97")


 merge:merge_clusters_by_entity_iou:97
n= 21
features= [(2100, 0.5770000219345093), (2348, -0.5770000219345093), (2427, -0.5770000219345093)]
tags:
movies:
  104 1091 Cruel & Unusual (Kay Scarpetta, #4) | 
  1196 2074 The Body Farm (Kay Scarpetta, #5) | 
  1281 2150 Body of Evidence (Kay Scarpetta, #2) | 
  1442 2296 Unnatural Exposure (Kay Scarpetta, #8) | 
  1504 2351 All That Remains (Kay Scarpetta, #3) | 
  1941 2745 From Potter's Field (Kay Scarpetta, #6) | 
  2345 3108 Cause of Death (Kay Scarpetta, #7) | 
  2717 3443 Portrait of a Killer: Jack the Ripper - Case Closed | 
  3477 4127 Black Notice (Kay Scarpetta, #10) | 
  4220 4797 Trace (Kay Scarpetta, #13) | 
  4455 5007 Predator (Kay Scarpetta, #14) | 
  4569 511 Postmortem (Kay Scarpetta, #1) | 
  4631 5166 Scarpetta (Kay Scarpetta, #16) | 
  4760 5282 Book of the Dead (Kay Scarpetta, #15) | 
  5439 5894 Red Mist (Kay Scarpetta, #19) | 
  6027 6422 The Scarpetta Factor (Kay Scarpetta, #17) | 
  6532 6878 The Bone Bed (Kay Sc

In [11]:
show_cluster("feature:646:neg")

KeyError: 'feature:646:neg'

In [13]:
def show_cluster(cid, limit=100):
    c = clusters.cluster_by_id[cid]
    print("\n", c.cluster_id)
    print("n=", c.entity_count)
    print("features=", list(zip(c.centroid.indices.tolist(), c.centroid.values.round(3).tolist())))
    print("tags:")
    for t in c.tags[:10]:
        print(f"  {t.name}: score={t.score:.3f}, count={t.count:.0f}")

    print("movies:")
    for row_idx in c.entity_indices[:limit]:
        row = meta.iloc[row_idx]
        print(" ", row_idx, row["item_id"], row["title"], "|", row.get("genres", ""))

show_cluster("feature:508:neg")
show_cluster("feature:646:neg")

KeyError: 'feature:508:neg'

In [35]:
clusters = cc.cluster_srp(
    srp,
    mode="dominant_signed",
    top_m=1,
    min_cluster_size=5,
    entity_tag_matrix=tags,
    tag_names=tag_names,
    top_k_tags=5,
    
)

In [36]:
largest = sorted(clusters.clusters, key=lambda c: c.entity_count, reverse=True)

for c in largest[:10]:
    print("\n", c.cluster_id, "n=", c.entity_count)
    print("tags:", ", ".join(f"{t.name}:{t.score:.3f}" for t in c.tags[:5]))

    for row_idx in c.entity_indices[:10]:
        row = meta.iloc[row_idx]
        print(" ", row["item_id"], row["title"], "|", row.get("genres", ""))



 feature:614:pos n= 17
tags: less than 300 ratings:0.546, animation:0.462, low budget:0.462, sci-fi:0.462, time travel:0.389
  1364 Zero Kelvin (Kjærlighetens kjøtere) (1995) | Action
  139 Target (1995) | Action|Drama
  1561 Wedding Bell Blues (1996) | Comedy
  1782 Little City (1998) | Comedy|Romance
  192 Show, The (1995) | Documentary
  2823 Spiders, The (Die Spinnen, 1. Teil: Der Goldene See) (1919) | Action|Drama
  2868 Fright Night Part II (1989) | Horror
  3047 Experience Preferred... But Not Essential (1982) | Drama
  3057 Where's Marlowe? (1999) | Comedy
  3154 Blood on the Sun (1945) | Drama|War

 feature:623:neg n= 13
tags: science:0.598, true story:0.399, genius:0.399, london:0.399, alfred hitchcock:0.399
  1040 Secret Agent, The (1996) | Drama
  1444 Guantanamera (1994) | Comedy
  1795 Callejón de los milagros, El (1995) | Drama
  1988 Hello Mary Lou: Prom Night II (1987) | Horror
  1990 Prom Night IV: Deliver Us From Evil (1992) | Horror
  200 Tie That Binds, The (1995)

In [5]:
clusters_iou = cc.cluster_srp(
    srp,
    mode="top_m_signed",
    top_m=1,
    min_cluster_size=5,
    activation_iou_threshold=0.5,
    entity_tag_matrix=tags,
    tag_names=tag_names,
    top_k_tags=5,
)

In [6]:
largest = sorted(clusters_iou.clusters, key=lambda c: c.entity_count, reverse=True)

for c in largest[:10]:
    print("\n", c.cluster_id, "n=", c.entity_count)
    print("tags:", ", ".join(f"{t.name}:{t.score:.3f}" for t in c.tags[:5]))

    for row_idx in c.entity_indices[:10]:
        row = meta.iloc[row_idx]
        print(" ", row["item_id"], row["title"], "|", row.get("genres", ""))



 feature:614:pos n= 17
tags: less than 300 ratings:0.546, animation:0.462, low budget:0.462, sci-fi:0.462, time travel:0.389
  1364 Zero Kelvin (Kjærlighetens kjøtere) (1995) | Action
  139 Target (1995) | Action|Drama
  1561 Wedding Bell Blues (1996) | Comedy
  1782 Little City (1998) | Comedy|Romance
  192 Show, The (1995) | Documentary
  2823 Spiders, The (Die Spinnen, 1. Teil: Der Goldene See) (1919) | Action|Drama
  2868 Fright Night Part II (1989) | Horror
  3047 Experience Preferred... But Not Essential (1982) | Drama
  3057 Where's Marlowe? (1999) | Comedy
  3154 Blood on the Sun (1945) | Drama|War

 feature:623:neg n= 13
tags: science:0.598, true story:0.399, genius:0.399, london:0.399, alfred hitchcock:0.399
  1040 Secret Agent, The (1996) | Drama
  1444 Guantanamera (1994) | Comedy
  1795 Callejón de los milagros, El (1995) | Drama
  1988 Hello Mary Lou: Prom Night II (1987) | Horror
  1990 Prom Night IV: Deliver Us From Evil (1992) | Horror
  200 Tie That Binds, The (1995)

In [9]:
clusters_top2 = cc.cluster_srp(
    srp,
    mode="top_m_signed",
    top_m=3,
    min_cluster_size=5,
    entity_tag_matrix=tags,
    tag_names=tag_names,
    top_k_tags=5,
)

print("top2 no merge:", len(clusters_top2.clusters))
print("top2 iou merge:", len(clusters_iou.clusters))


top2 no merge: 950
top2 iou merge: 946


In [10]:
largest_iou = sorted(clusters_iou.clusters, key=lambda c: c.entity_count, reverse=True)

for c in largest_iou[:10]:
    print("\n", c.cluster_id, "n=", c.entity_count, "features=", len(c.centroid.indices))
    print("centroid:", list(zip(c.centroid.indices.tolist(), c.centroid.values.round(3).tolist())))
    print("tags:", ", ".join(f"{t.name}:{t.score:.3f}" for t in c.tags[:5]))

    for row_idx in c.entity_indices[:10]:
        row = meta.iloc[row_idx]
        print(" ", row["title"], "|", row.get("genres", ""))



 feature:641:neg n= 54 features= 1
centroid: [(641, -1.0)]
tags: Drama:0.318, Children's:0.216, Horror:0.201, Comedy:0.180, Action:0.148
  American Strays (1996) | Action
  Children of the Corn IV: The Gathering (1996) | Horror
  Steal Big, Steal Little (1995) | Comedy
  Anna (1996) | Drama
  Zeus and Roxanne (1997) | Children's
  Temptress Moon (Feng Yue) (1996) | Romance
  Broken English (1996) | Drama
  Warriors of Virtue (1997) | Action|Adventure|Children's|Fantasy
  Head Above Water (1996) | Comedy|Thriller
  Simple Wish, A (1997) | Children's|Fantasy

 feature:218:pos n= 41 features= 1
centroid: [(218, 1.0)]
tags: Horror:0.279, Comedy:0.273, Drama:0.243, Romance:0.217, Action:0.171
  Surviving Picasso (1996) | Drama
  Amityville II: The Possession (1982) | Horror
  Message to Love: The Isle of Wight Festival (1996) | Documentary
  B*A*P*S (1997) | Comedy
  Critical Care (1997) | Comedy
  Chairman of the Board (1998) | Comedy
  Midaq Alley (Callejón de los milagros, El) (1995) | 

In [14]:
merged = cc.cluster_srp(
    srp,
    mode="dominant_signed",
    top_m=3,
    min_cluster_size=5,
    entity_tag_matrix=tags,
    tag_names=tag_names,
    top_k_tags=5,
    tag_similarity_threshold=0.5,
)

print(len(clusters.clusters), "->", len(merged.clusters))


58 -> 29


In [16]:
largest = sorted(clusters.clusters, key=lambda c: c.entity_count, reverse=True)

for c in largest[:10]:
    print("\n", c.cluster_id, "n=", c.entity_count)
    print("tags:", ", ".join(f"{t.name}:{t.score:.3f}" for t in c.tags[:5]))

    for row_idx in c.entity_indices[:10]:
        row = meta.iloc[row_idx]
        print(" ", row["item_id"], row["title"], "|", row.get("genres", ""))



 feature:614:pos n= 17
tags: Action:0.448, Drama:0.293, Comedy:0.182, Thriller:0.146, Animation:0.142
  1364 Zero Kelvin (Kjærlighetens kjøtere) (1995) | Action
  139 Target (1995) | Action|Drama
  1561 Wedding Bell Blues (1996) | Comedy
  1782 Little City (1998) | Comedy|Romance
  192 Show, The (1995) | Documentary
  2823 Spiders, The (Die Spinnen, 1. Teil: Der Goldene See) (1919) | Action|Drama
  2868 Fright Night Part II (1989) | Horror
  3047 Experience Preferred... But Not Essential (1982) | Drama
  3057 Where's Marlowe? (1999) | Comedy
  3154 Blood on the Sun (1945) | Drama|War

 feature:623:neg n= 13
tags: Drama:0.408, Comedy:0.296, Horror:0.248, Thriller:0.237, Western:0.217
  1040 Secret Agent, The (1996) | Drama
  1444 Guantanamera (1994) | Comedy
  1795 Callejón de los milagros, El (1995) | Drama
  1988 Hello Mary Lou: Prom Night II (1987) | Horror
  1990 Prom Night IV: Deliver Us From Evil (1992) | Horror
  200 Tie That Binds, The (1995) | Thriller
  2207 Jamaica Inn (1939

In [22]:
for th in [0.1, 0.3, 0.5, 0.7, 0.9]:
    c = cc.cluster_srp(
        srp,
        mode="top_m_signed",
        top_m=3,
        min_cluster_size=10,
        activation_iou_threshold=th,
        entity_tag_matrix=tags,
        tag_names=tag_names,
        top_k_tags=5,
    )
    print("top3 iou", th, "clusters", len(c.clusters), "largest", max(x.entity_count for x in c.clusters), "smallest", min(x.entity_count for x in c.clusters))


top3 iou 0.1 clusters 145 largest 54 smallest 10
top3 iou 0.3 clusters 149 largest 54 smallest 10
top3 iou 0.5 clusters 149 largest 54 smallest 10
top3 iou 0.7 clusters 149 largest 54 smallest 10
top3 iou 0.9 clusters 149 largest 54 smallest 10


In [23]:
largest = sorted(c.clusters, key=lambda c: c.entity_count, reverse=True)

for c in largest[:10]:
    print("\n", c.cluster_id, "n=", c.entity_count)
    print("tags:", ", ".join(f"{t.name}:{t.score:.3f}" for t in c.tags[:5]))

    for row_idx in c.entity_indices[:10]:
        row = meta.iloc[row_idx]
        print(" ", row["item_id"], row["title"], "|", row.get("genres", ""))



 feature:641:neg n= 54
tags: Drama:0.281, Children's:0.161, Comedy:0.156, Horror:0.149, Fantasy:0.117
  1102 American Strays (1996) | Action
  1105 Children of the Corn IV: The Gathering (1996) | Horror
  119 Steal Big, Steal Little (1995) | Comedy
  1316 Anna (1996) | Drama
  1426 Zeus and Roxanne (1997) | Children's
  1514 Temptress Moon (Feng Yue) (1996) | Romance
  1519 Broken English (1996) | Drama
  1525 Warriors of Virtue (1997) | Action|Adventure|Children's|Fantasy
  1565 Head Above Water (1996) | Comedy|Thriller
  1583 Simple Wish, A (1997) | Children's|Fantasy

 feature:218:pos n= 41
tags: Comedy:0.236, Drama:0.215, Horror:0.206, Romance:0.161, Action:0.134
  1044 Surviving Picasso (1996) | Drama
  1326 Amityville II: The Possession (1982) | Horror
  1420 Message to Love: The Isle of Wight Festival (1996) | Documentary
  1490 B*A*P*S (1997) | Comedy
  1677 Critical Care (1997) | Comedy
  1679 Chairman of the Board (1998) | Comedy
  1741 Midaq Alley (Callejón de los milagros,